# H-007 · Windows (sequential)

Sequential screens after the book is frozen:

1. Screening `L` ∈ {252, 504} — only if `BOOK_STAR` is `quarterly` or `12m6m`
2. OLS / beta ∈ {252, 504}
3. ADF ∈ {252, 504}
4. Entry z ∈ {1.5, 2.0}
5. Z window ∈ {60, 90}

Do not open the sealed holdout in this notebook.

Next: run H-008.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    overlay_kalman_hedge,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds

VERSION = "v2"
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
LEDGER_PATH = os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json")
cuts = load_s2_cuts(root=ROOT, version=VERSION)
stack = load_star_stack(STAR_PATH)
print("yahoo_drop_rate", cuts.get("yahoo_drop_rate"))
print("stack keys", sorted(stack.keys()))

from backtest.s2_coint.research import rebalance_path
HYP_ID = "H-007"

require_star("POOL_STAR", stack.get("POOL_STAR"))
require_star("FDR_STAR", stack.get("FDR_STAR"))
require_star("BOOK_STAR", stack.get("BOOK_STAR"))
require_star("N_STAR", stack.get("N_STAR"))
PAIRS_STAR = list(stack.get("PAIRS_STAR") or [])
if not PAIRS_STAR:
    raise ValueError("PAIRS_STAR empty")
BAR = str(stack.get("BAR_STAR") or "1d")
train, full = load_universe_panels(None, BAR, PAIRS_STAR, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(
    BAR,
    ols_days=stack.get("OLS_WINDOW_STAR"),
    z_days=stack.get("Z_WINDOW_STAR"),
    adf_days=stack.get("ADF_WINDOW_STAR"),
)
train = overlay_ols_hedge(train, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                          hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
full = overlay_ols_hedge(full, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                         hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    from backtest.s2_coint.research import split_is_oos
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS", len(is_panel), "pairs", len(PAIRS_STAR))



## 1. Optional L screen (rotate only)


In [ ]:
BOOK_STAR = str(stack["BOOK_STAR"])
panel = is_panel.copy()
dates = pd.DatetimeIndex(pd.to_datetime(panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
display(fold_table(folds))

if BOOK_STAR in {"quarterly", "12m6m"}:
    reb = pd.read_csv(rebalance_path(BAR, version=VERSION, root=ROOT), parse_dates=["rebalance_date", "effective_date"])
    sched = BOOK_STAR
    L_ARMS = (252, 504)
    register_hypothesis_arms(HYP_ID + "_L", [str(x) for x in L_ARMS], overwrite=True, path=LEDGER_PATH)
    fold_parts, full_parts = [], []
    for L in L_ARMS:
        sub = reb.loc[(reb["schedule"] == sched) & (reb["lookback"] == L)].copy()
        last = sub.loc[sub["rebalance_date"] == sub["rebalance_date"].max()].sort_values("rank")
        ids = [str(p) for p in last["pair_id"].head(int(stack["N_STAR"]))]
        p = panel.loc[panel["pair_id"].astype(str).isin(set(ids))].copy()
        cfg = {str(L): config_from_stack(stack)}
        fold_parts.append(fold_val_metrics(p, folds, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
        full_parts.append(full_is_metrics(p, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    fold_L = pd.concat(fold_parts, ignore_index=True)
    full_L = pd.concat(full_parts, ignore_index=True)
    plot_fold_boxplots(fold_L, title="H-007 L fold-val")
    plt.show()
    display(arm_selection_table(fold_L, full_L))
    L_STAR = ""  # type: "252" | "504"
    require_star("L_STAR", L_STAR)
    stack["L_STAR"] = int(L_STAR)
    save_star_stack(STAR_PATH, stack)
    print("L_STAR", L_STAR)
else:
    print("BOOK_STAR=freeze — skip L screen")
    stack["L_STAR"] = None
    save_star_stack(STAR_PATH, stack)



## 2. OLS / beta window


In [ ]:
OLS_ARMS = (252, 504)
register_hypothesis_arms(HYP_ID + "_OLS", [str(x) for x in OLS_ARMS], overwrite=True, path=LEDGER_PATH)
configs = {}
for w in OLS_ARMS:
    lb = lookbacks_for_bar(BAR, ols_days=w, z_days=stack.get("Z_WINDOW_STAR"), adf_days=stack.get("ADF_WINDOW_STAR"))
    p = overlay_ols_hedge(panel, ols_window=lb["ols_window"], z_window=lb["z_window"],
                          hl_window=lb["hl_window"], adf_window=lb["adf_window"])
    configs[str(w)] = (p, config_from_stack(stack, ols_window=lb["ols_window"]))
fold_parts, full_parts = [], []
for arm, (p, cfg) in configs.items():
    fold_parts.append(fold_val_metrics(p, folds, {arm: cfg}, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    full_parts.append(full_is_metrics(p, {arm: cfg}, hyp_id=HYP_ID, s1_weekly=s1_weekly))
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-007 OLS fold-val")
plt.show()
display(arm_selection_table(fold_df, full_is_df))
OLS_WINDOW_STAR = ""  # type: "252" | "504"
require_star("OLS_WINDOW_STAR", OLS_WINDOW_STAR)
stack["OLS_WINDOW_STAR"] = int(OLS_WINDOW_STAR)
save_star_stack(STAR_PATH, stack)



## 3. ADF window


In [ ]:
ADF_ARMS = (252, 504)
register_hypothesis_arms(HYP_ID + "_ADF", [str(x) for x in ADF_ARMS], overwrite=True, path=LEDGER_PATH)
lb_ols = lookbacks_for_bar(BAR, ols_days=stack["OLS_WINDOW_STAR"])
fold_parts, full_parts = [], []
for w in ADF_ARMS:
    lb = lookbacks_for_bar(BAR, ols_days=stack["OLS_WINDOW_STAR"], adf_days=w, z_days=stack.get("Z_WINDOW_STAR"))
    p = overlay_ols_hedge(panel, ols_window=lb["ols_window"], z_window=lb["z_window"],
                          hl_window=lb["hl_window"], adf_window=lb["adf_window"])
    cfg = {str(w): config_from_stack(stack, adf_window=lb["adf_window"])}
    fold_parts.append(fold_val_metrics(p, folds, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    full_parts.append(full_is_metrics(p, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-007 ADF fold-val")
plt.show()
display(arm_selection_table(fold_df, full_is_df))
ADF_WINDOW_STAR = ""  # type: "252" | "504"
require_star("ADF_WINDOW_STAR", ADF_WINDOW_STAR)
stack["ADF_WINDOW_STAR"] = int(ADF_WINDOW_STAR)
save_star_stack(STAR_PATH, stack)



## 4. Entry z


In [ ]:
EZ_ARMS = (1.5, 2.0)
register_hypothesis_arms(HYP_ID + "_EZ", [str(x) for x in EZ_ARMS], overwrite=True, path=LEDGER_PATH)
lb = lookbacks_for_bar(BAR, ols_days=stack["OLS_WINDOW_STAR"], adf_days=stack["ADF_WINDOW_STAR"],
                       z_days=stack.get("Z_WINDOW_STAR"))
p = overlay_ols_hedge(panel, ols_window=lb["ols_window"], z_window=lb["z_window"],
                      hl_window=lb["hl_window"], adf_window=lb["adf_window"])
configs = {str(z): config_from_stack(stack, entry_z=float(z)) for z in EZ_ARMS}
fold_df = fold_val_metrics(p, folds, configs, hyp_id=HYP_ID, s1_weekly=s1_weekly)
full_is_df = full_is_metrics(p, configs, hyp_id=HYP_ID, s1_weekly=s1_weekly)
plot_fold_boxplots(fold_df, title="H-007 entry-z fold-val")
plt.show()
display(arm_selection_table(fold_df, full_is_df))
ENTRY_Z_STAR = ""  # type: "1.5" | "2.0"
require_star("ENTRY_Z_STAR", ENTRY_Z_STAR)
stack["ENTRY_Z_STAR"] = float(ENTRY_Z_STAR)
save_star_stack(STAR_PATH, stack)



## 5. Z window


In [ ]:
Z_ARMS = (60, 90)
register_hypothesis_arms(HYP_ID + "_Z", [str(x) for x in Z_ARMS], overwrite=True, path=LEDGER_PATH)
fold_parts, full_parts = [], []
for w in Z_ARMS:
    lb = lookbacks_for_bar(BAR, ols_days=stack["OLS_WINDOW_STAR"], adf_days=stack["ADF_WINDOW_STAR"], z_days=w)
    p = overlay_ols_hedge(panel, ols_window=lb["ols_window"], z_window=lb["z_window"],
                          hl_window=lb["hl_window"], adf_window=lb["adf_window"])
    cfg = {str(w): config_from_stack(stack, z_window=lb["z_window"])}
    fold_parts.append(fold_val_metrics(p, folds, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    full_parts.append(full_is_metrics(p, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-007 z-window fold-val")
plt.show()
display(arm_selection_table(fold_df, full_is_df))
Z_WINDOW_STAR = ""  # type: "60" | "90"
require_star("Z_WINDOW_STAR", Z_WINDOW_STAR)
stack["Z_WINDOW_STAR"] = int(Z_WINDOW_STAR)
save_star_stack(STAR_PATH, stack)
print("windows frozen", {k: stack[k] for k in ("L_STAR","OLS_WINDOW_STAR","ADF_WINDOW_STAR","ENTRY_Z_STAR","Z_WINDOW_STAR")})



## 6. Notes for next hyp

Next: run H-008.
